# 02 – Silver Layer: EMS Incident Dispatch Cleaning

**Purpose:** Turn raw Bronze data into clean, typed, trustworthy data.

**Input:** Latest snapshot in `bronze/ems_incident_dispatch/`
**Output:** Silver table used for the Gold analysis (response time by borough and time period, and by call type and severity).

**Silver rules:** fix types, handle missing values, standardize categories, remove invalid or duplicate rows, and document every rule. The cleaning rules live in `src/silver_rules.py`, where they are covered by automated tests.

## 1. Load configuration

Paths and settings are stored in `config/dev.yaml` and `config/prod.yaml`, not in the code. Change `ENV` to `"prod"` to run on the full dataset.

In [1]:
import sys
sys.path.append("../src")          # lets the notebook import files from src/

from config_loader import load_config

ENV = "prod"                        # "dev" = 2,000-row sample, "prod" = full data
config = load_config(f"../config/{ENV}.yaml")

print(config)

{'environment': 'prod', 'spark': {'driver_memory': '4g', 'cores': 2, 'shuffle_partitions': 400}, 'source': {'dataset_name': 'ems_incident_dispatch', 'dataset_id': '76xm-jjuj', 'input_path': '/home/ubuntu/spark-notebooks/ems-pipeline/data_sample/EMS_Incident_Dispatch_Data.csv'}, 'bronze': {'root_path': '/home/ubuntu/spark-notebooks/ems-pipeline/data/bronze'}, 'silver': {'root_path': '/home/ubuntu/spark-notebooks/ems-pipeline/data/silver'}}


## 2. Start Spark

Create the Spark session, which is the entry point for all Spark operations. Memory, cores and shuffle partitions come from the config. `getOrCreate()` reuses a running session, so **restart the kernel** after changing these settings.

In [2]:
from pyspark.sql import SparkSession

spark_cfg = config["spark"]

spark = (
    SparkSession.builder
    .appName("ems_pipeline")
    .master(f"local[{spark_cfg['cores']}]")
    .config("spark.driver.memory", spark_cfg["driver_memory"])
    .config("spark.sql.shuffle.partitions", spark_cfg["shuffle_partitions"])
    .getOrCreate()
)

print(spark.version)
print("Master:", spark.sparkContext.master)
print("Driver memory:", spark.sparkContext.getConf().get("spark.driver.memory"))

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/10 12:03:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


3.5.0
Master: local[2]
Driver memory: 4g


## Performance measurement helper

Spark is lazy: transformations like `withColumn` take no time, because the work only happens at an **action** (`count`, `collect`, `write`). We wrap the actions in `timed(...)`, and each printed time covers the whole chain of work behind that action.

In [3]:
import time
from contextlib import contextmanager

@contextmanager
def timed(label):
    start = time.time()
    yield
    print(f"[{label}] {time.time() - start:.1f} s")

## 3. Read the latest Bronze snapshot

Bronze snapshots are stored in folders named `snapshot_date=YYYY-MM-DD`. Sorting the folder names gives us the newest one. We read it as Parquet, so all columns are still raw strings.

In [4]:
import os, glob

bronze_root = os.path.abspath(os.path.join(
    "..", config["bronze"]["root_path"], config["source"]["dataset_name"]
))

snapshots = sorted(glob.glob(os.path.join(bronze_root, "snapshot_date=*")))
print("Snapshots found:", [os.path.basename(s) for s in snapshots])

latest_snapshot = snapshots[-1]
print("Using:", latest_snapshot)

bronze_df = spark.read.parquet(latest_snapshot)
total_rows = bronze_df.count()
print("Rows:", total_rows)

Snapshots found: ['snapshot_date=2026-11-02']
Using: /home/ubuntu/spark-notebooks/ems-pipeline/data/bronze/ems_incident_dispatch/snapshot_date=2026-11-02


Rows: 30389767


## 4. Profile missing values (the "before" picture)

Missing values can appear as a real null, the text `NULL`, or an empty string. This table counts all three for every source column. On the full data they are all real nulls.

In [5]:
import pandas as pd
from pyspark.sql import functions as F

cols = [c for c in bronze_df.columns if not c.startswith("_")]

aggs = []
for c in cols:
    aggs.append(F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(f"{c}__real_null"))
    aggs.append(F.sum(F.when(F.trim(F.col(c)).isin("NULL", ""), 1).otherwise(0)).alias(f"{c}__null_text"))

with timed("Profile missing values"):
    counts = bronze_df.agg(*aggs).collect()[0].asDict()

profile = pd.DataFrame({
    "column": cols,
    "real_null": [counts[f"{c}__real_null"] for c in cols],
    "null_text_or_empty": [counts[f"{c}__null_text"] for c in cols],
})
profile["missing_total"] = profile["real_null"] + profile["null_text_or_empty"]
profile["missing_pct"] = (100 * profile["missing_total"] / total_rows).round(1)

pd.set_option("display.max_rows", 100)
profile.sort_values("missing_total", ascending=False).reset_index(drop=True)

26/11/10 12:04:13 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
[Stage 4:=====================================================>   (17 + 1) / 18]

[Profile missing values] 33.9 s


,column,real_null,null_text_or_empty,missing_total,missing_pct
0,FIRST_HOSP_ARRIVAL_DATETIME,9971312,0,9971312,32.8
1,FIRST_TO_HOSP_DATETIME,9799315,0,9799315,32.2
2,INCIDENT_RESPONSE_SECONDS_QY,1128836,0,1128836,3.7
3,INCIDENT_TRAVEL_TM_SECONDS_QY,1123304,0,1123304,3.7
4,FIRST_ON_SCENE_DATETIME,1122848,0,1122848,3.7
5,COMMUNITYSCHOOLDISTRICT,614342,0,614342,2.0
6,ZIPCODE,601029,0,601029,2.0
7,COMMUNITYDISTRICT,600069,0,600069,2.0
8,POLICEPRECINCT,599974,0,599974,2.0
9,CITYCOUNCILDISTRICT,599974,0,599974,2.0


## 5. Silver transformations

The cleaning rules are called from `src/silver_rules.py` (tested by 11 unit tests that run in GitHub Actions). Order: convert types, add flags and derived fields, reject unusable rows, check duplicates, flag logical problems.

**Duplicate check:** the grain of Silver is one record per `INCIDENT_ID`. A cheap one-column check runs first. The expensive full deduplication (a shuffle of all rows) runs **only if duplicates exist**.

In [6]:
from pyspark.sql import functions as F
from silver_rules import (
    convert_types, add_flags_and_derived, split_rejected,
    dedupe_by_incident_id, flag_logical_issues, logical_check_expressions,
)

typed_df = convert_types(bronze_df)
silver_df = add_flags_and_derived(typed_df)
accepted_df, rejected_df = split_rejected(silver_df)

# Duplicate check on ONE column (cheap) instead of shuffling every column of every row
with timed("Duplicate check (INCIDENT_ID only)"):
    rows_accepted = accepted_df.count()
    distinct_ids = accepted_df.select("INCIDENT_ID").distinct().count()
dup_count = rows_accepted - distinct_ids
print("Accepted rows:", rows_accepted, "| Duplicate incident IDs:", dup_count)

# Full deduplication only if duplicates exist
deduped_df = dedupe_by_incident_id(accepted_df) if dup_count > 0 else accepted_df
clean_df = flag_logical_issues(deduped_df)

checks = logical_check_expressions()
rows_in = total_rows

26/11/10 12:06:05 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/10 12:06:05 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
[Stage 12:===========================================>              (3 + 1) / 4]

[Duplicate check (INCIDENT_ID only)] 48.9 s
Accepted rows: 30389767 | Duplicate incident IDs: 0


### Rules applied in Silver

- **Types:** text dates become timestamps, `942.0` becomes `942`, ZIP codes stay 5-digit text.
- **Flags (kept, not deleted):** `is_valid_dispatch_response`, `is_valid_incident_response`, and `has_data_issue` (out-of-order events or negative durations).
- **Reject:** only records with no `INCIDENT_ID` or no incident time.
- **Deduplicate:** one record per `INCIDENT_ID`.
- **Borough:** `RICHMOND / STATEN ISLAND` becomes `STATEN ISLAND` in `borough_std`. The few records with `UNKNOWN` are kept as `UNKNOWN`, not guessed.

## 6. Write the Silver outputs

**Silver table:** Parquet, partitioned by `incident_year` (one folder per year). `repartition("incident_year")` before the write groups each year into one task, so each year is written as about one file instead of many tiny files.

**Overwrite is correct here:** Silver is rebuilt from a Bronze snapshot, so a re-run replaces the old output with the same result (idempotent).

**Lineage columns:** `_bronze_snapshot_date` and `_silver_processed_at` record which Bronze snapshot this Silver data came from and when.

**Rejected records:** saved separately with a reason, so they stay visible and countable.

In [7]:
silver_root = os.path.abspath(os.path.join(
    "..", config["silver"]["root_path"], config["source"]["dataset_name"]
))
silver_data_path = os.path.join(silver_root, "data")
rejected_path = os.path.join(silver_root, "rejected")

bronze_snapshot_date = os.path.basename(latest_snapshot).split("=")[1]

silver_out = (
    clean_df
    .withColumn("_bronze_snapshot_date", F.lit(bronze_snapshot_date))
    .withColumn("_silver_processed_at", F.current_timestamp())
)

rejected_out = rejected_df.withColumn(
    "_reject_reason", F.lit("missing incident id or incident time")
)

with timed("Silver write"):
    (silver_out
        .repartition("incident_year")
        .write.mode("overwrite")
        .partitionBy("incident_year")
        .parquet(silver_data_path))

rejected_out.write.mode("overwrite").parquet(rejected_path)

print("Silver table written to:", silver_data_path)
print("Rejected records written to:", rejected_path)

[Silver write] 292.3 s


[Stage 19:====================================================>   (17 + 1) / 18]

Silver table written to: /home/ubuntu/spark-notebooks/ems-pipeline/data/silver/ems_incident_dispatch/data
Rejected records written to: /home/ubuntu/spark-notebooks/ems-pipeline/data/silver/ems_incident_dispatch/rejected


## 7. Quality summary (computed from the written Silver data)

The expensive transformation runs **once**, in the write cell above. The quality numbers are computed by reading back the Parquet files that were written, which is far cheaper than re-running the whole pipeline for every count.

Two real checks at the end: (1) row counts must add up, and (2) every `INCIDENT_ID` in Silver must be unique, which proves the grain.

In [8]:
import pandas as pd

silver_check = spark.read.parquet(silver_data_path)

with timed("Quality summary"):
    rows_rejected = rows_in - rows_accepted
    rows_out = silver_check.count()

    flag_counts = silver_check.agg(
        F.sum(F.when(~F.col("is_valid_dispatch_response"), 1).otherwise(0)).alias("invalid_dispatch"),
        F.sum(F.when(~F.col("is_valid_incident_response"), 1).otherwise(0)).alias("invalid_incident"),
        F.sum(F.when(F.col("has_data_issue"), 1).otherwise(0)).alias("data_issue"),
    ).collect()[0].asDict()

    check_counts = silver_check.agg(
        *[F.sum(F.when(expr, 1).otherwise(0)).alias(name) for name, expr in checks.items()]
    ).collect()[0].asDict()

with timed("Grain check on written Silver"):
    distinct_ids_out = silver_check.select("INCIDENT_ID").distinct().count()

assert rows_out == rows_in - rows_rejected - dup_count, "Row counts do not add up!"
assert distinct_ids_out == rows_out, "Silver has duplicate INCIDENT_IDs!"

print("Records failing each logical check:")
for name, n in check_counts.items():
    print(f"  {name}: {n}")

quality_summary = pd.DataFrame([
    ("Rows in (Bronze)", rows_in),
    ("Rejected (no ID or no incident time)", rows_rejected),
    ("Duplicate incident IDs removed", dup_count),
    ("Rows out (Silver)", rows_out),
    ("Flagged: invalid dispatch response", flag_counts["invalid_dispatch"]),
    ("Flagged: invalid incident response", flag_counts["invalid_incident"]),
    ("Flagged: logical data issue", flag_counts["data_issue"]),
], columns=["metric", "count"])
quality_summary

[Quality summary] 8.2 s


26/11/10 12:14:40 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/10 12:14:40 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
[Stage 32:===========================================>              (3 + 1) / 4]

[Grain check on written Silver] 21.1 s
Records failing each logical check:
  assignment_before_incident: 176
  on_scene_before_assignment: 450
  hospital_arrival_before_departure: 13741
  close_before_incident: 1140
  negative_response_seconds: 0


,metric,count
0,Rows in (Bronze),30389767
1,Rejected (no ID or no incident time),0
2,Duplicate incident IDs removed,0
3,Rows out (Silver),30389767
4,Flagged: invalid dispatch response,242125
5,Flagged: invalid incident response,1128836
6,Flagged: logical data issue,15290


## 8. Save the quality summary

The before/after counts are saved as a JSON file, like the Bronze run log, so every Silver run leaves auditable evidence of what entered, what was rejected, and what was flagged.

In [9]:
import json
from datetime import datetime, timezone

quality_dir = os.path.join(silver_root, "_quality_log")
os.makedirs(quality_dir, exist_ok=True)

quality_record = {
    "environment": config["environment"],
    "bronze_snapshot_date": bronze_snapshot_date,
    "processed_at_utc": datetime.now(timezone.utc).isoformat(),
    "metrics": {row["metric"]: int(row["count"]) for _, row in quality_summary.iterrows()},
    "logical_checks": {k: int(v) for k, v in check_counts.items()},
}

quality_file = os.path.join(quality_dir, f"silver_quality_{bronze_snapshot_date}.json")
with open(quality_file, "w") as f:
    json.dump(quality_record, f, indent=2)

print(json.dumps(quality_record, indent=2))

{
  "environment": "prod",
  "bronze_snapshot_date": "2026-11-02",
  "processed_at_utc": "2026-11-10T12:15:46.717251+00:00",
  "metrics": {
    "Rows in (Bronze)": 30389767,
    "Rejected (no ID or no incident time)": 0,
    "Duplicate incident IDs removed": 0,
    "Rows out (Silver)": 30389767,
    "Flagged: invalid dispatch response": 242125,
    "Flagged: invalid incident response": 1128836,
    "Flagged: logical data issue": 15290
  },
  "logical_checks": {
    "assignment_before_incident": 176,
    "on_scene_before_assignment": 450,
    "hospital_arrival_before_departure": 13741,
    "close_before_incident": 1140,
    "negative_response_seconds": 0
  }
}


### Verify the Silver output

Read the Silver table back from disk. The row count must equal "Rows out (Silver)", the types must be correct, and one partition folder per year must exist.

In [10]:
check_silver = spark.read.parquet(silver_data_path)

print("Silver rows on disk:", check_silver.count())
print("Rows out per quality summary:", rows_out)
print("Rejected rows on disk:", spark.read.parquet(rejected_path).count())
print("Partition folders:", sorted(d for d in os.listdir(silver_data_path) if d.startswith("incident_year=")))

check_silver.select("INCIDENT_ID", "INCIDENT_DATETIME", "incident_year", "borough_std",
                    "is_valid_incident_response", "has_data_issue").printSchema()

Silver rows on disk: 30389767
Rows out per quality summary: 30389767
Rejected rows on disk: 0
Partition folders: ['incident_year=2005', 'incident_year=2006', 'incident_year=2007', 'incident_year=2008', 'incident_year=2009', 'incident_year=2010', 'incident_year=2011', 'incident_year=2012', 'incident_year=2013', 'incident_year=2014', 'incident_year=2015', 'incident_year=2016', 'incident_year=2017', 'incident_year=2018', 'incident_year=2019', 'incident_year=2020', 'incident_year=2021', 'incident_year=2022', 'incident_year=2023', 'incident_year=2024', 'incident_year=2025', 'incident_year=2026']
root
 |-- INCIDENT_ID: string (nullable = true)
 |-- INCIDENT_DATETIME: timestamp (nullable = true)
 |-- incident_year: integer (nullable = true)
 |-- borough_std: string (nullable = true)
 |-- is_valid_incident_response: boolean (nullable = true)
 |-- has_data_issue: boolean (nullable = true)



## 9. Performance evidence

**Execution plan.** `explain()` shows how Spark runs the transformations. An `Exchange` line is a **shuffle** (data moved between partitions), usually the most expensive operation. With no duplicate IDs, the full deduplication is skipped, so the plan should contain no deduplication shuffle.

In [11]:
clean_df.explain()

== Physical Plan ==
*(1) Project [INCIDENT_ID#0, INCIDENT_DATETIME#518, INITIAL_CALL_TYPE#2, INITIAL_SEVERITY_LEVEL_CODE#519, FINAL_CALL_TYPE#4, FINAL_SEVERITY_LEVEL_CODE#520, FIRST_ASSIGNMENT_DATETIME#521, VALID_DISPATCH_RSPNS_TIME_INDC#7, DISPATCH_RESPONSE_SECONDS_QY#522, FIRST_ACTIVATION_DATETIME#523, FIRST_ON_SCENE_DATETIME#524, VALID_INCIDENT_RSPNS_TIME_INDC#11, INCIDENT_RESPONSE_SECONDS_QY#525, INCIDENT_TRAVEL_TM_SECONDS_QY#526, FIRST_TO_HOSP_DATETIME#527, FIRST_HOSP_ARRIVAL_DATETIME#528, INCIDENT_CLOSE_DATETIME#529, HELD_INDICATOR#17, INCIDENT_DISPOSITION_CODE#18, BOROUGH#19, INCIDENT_DISPATCH_AREA#20, ZIPCODE#570, POLICEPRECINCT#530, CITYCOUNCILDISTRICT#531, ... 19 more fields]
+- *(1) Project [INCIDENT_ID#0, gettimestamp(INCIDENT_DATETIME#1, MM/dd/yyyy hh:mm:ss a, TimestampType, Some(Etc/UTC), false) AS INCIDENT_DATETIME#518, INITIAL_CALL_TYPE#2, cast(cast(INITIAL_SEVERITY_LEVEL_CODE#3 as double) as int) AS INITIAL_SEVERITY_LEVEL_CODE#519, FINAL_CALL_TYPE#4, cast(cast(FINAL_SE

**Output size and file layout.** Too many tiny Parquet files slow down every later read, so we check how many files were written and how big they are.

In [12]:
files = glob.glob(os.path.join(silver_data_path, "incident_year=*", "*.parquet"))
size_gb = sum(os.path.getsize(f) for f in files) / 1e9
print(len(files), "parquet files,", round(size_gb, 2), "GB")
print("Average file size:", round(size_gb * 1000 / max(len(files), 1), 1), "MB")
print("Year partitions:", len([d for d in os.listdir(silver_data_path) if d.startswith("incident_year=")]))

22 parquet files, 1.93 GB
Average file size: 87.9 MB
Year partitions: 22
